In [0]:
# Databricks notebook source

# COMMAND ----------
# Configuración

CATALOG_NAME    = "workspace"
SCHEMA_NAME     = "default"
TABLE_NAME      = "bronze_retail_sales"
FULL_TABLE_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
SOURCE_CSV_PATH = "/Volumes/workspace/default/raw_files/retail_store_sales.csv"

# COMMAND ----------
# Verificación de schema

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")

# COMMAND ----------
# Verificación de archivo fuente

import re
import os

files = dbutils.fs.ls(os.path.dirname(SOURCE_CSV_PATH))
found = [f.name for f in files if f.name == os.path.basename(SOURCE_CSV_PATH)]
if not found:
    raise FileNotFoundError(f"Archivo no encontrado: {SOURCE_CSV_PATH}")
print(f"✓ Archivo encontrado: {SOURCE_CSV_PATH}")

# COMMAND ----------
# Lectura CSV

import pandas as pd

pdf = pd.read_csv(SOURCE_CSV_PATH)
print(f"Filas: {len(pdf):,} | Columnas: {list(pdf.columns)}")

# COMMAND ----------
# Metadatos técnicos

from datetime import datetime, timezone

ingestion_ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
pdf["_ingestion_timestamp"] = ingestion_ts
pdf["_source_file"]         = SOURCE_CSV_PATH
pdf["_source_row_count"]    = len(pdf)

# COMMAND ----------
# Escritura Bronze

def clean_col(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name).strip('_').lower()

pdf.columns = [clean_col(c) for c in pdf.columns]

sdf = spark.createDataFrame(pdf)

(
    sdf.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(FULL_TABLE_NAME)
)

print(f"✓ Tabla escrita: {FULL_TABLE_NAME}")
print(f"Columnas finales: {list(pdf.columns)}")

# COMMAND ----------
# Validaciones

count_csv   = len(pdf)
count_delta = spark.table(FULL_TABLE_NAME).count()

print(f"Filas CSV   : {count_csv:,}")
print(f"Filas Delta : {count_delta:,}")

assert count_delta == count_csv, f"Discrepancia: {abs(count_delta - count_csv)} filas"
print("✓ Conteo coincide")

display(spark.table(FULL_TABLE_NAME).limit(5))

spark.sql(f"DESCRIBE TABLE EXTENDED {FULL_TABLE_NAME}").show(truncate=False)

✓ Archivo encontrado: /Volumes/workspace/default/raw_files/retail_store_sales.csv
Filas: 12,575 | Columnas: ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied']
✓ Tabla escrita: workspace.default.bronze_retail_sales
Columnas finales: ['transaction_id', 'customer_id', 'category', 'item', 'price_per_unit', 'quantity', 'total_spent', 'payment_method', 'location', 'transaction_date', 'discount_applied', 'ingestion_timestamp', 'source_file', 'source_row_count']
Filas CSV   : 12,575
Filas Delta : 12,575
✓ Conteo coincide


transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,ingestion_timestamp,source_file,source_row_count
TXN_3646423,CUST_24,Computers and electric accessories,Item_18_CEA,30.5,7.0,213.5,Credit Card,Online,2022-01-26,true,2026-04-29T07:27:21Z,/Volumes/workspace/default/raw_files/retail_store_sales.csv,12575
TXN_6605932,CUST_20,Beverages,Item_16_BEV,27.5,10.0,275.0,Credit Card,Online,2023-08-15,null,2026-04-29T07:27:21Z,/Volumes/workspace/default/raw_files/retail_store_sales.csv,12575
TXN_7654337,CUST_01,Furniture,Item_20_FUR,33.5,4.0,134.0,Digital Wallet,In-store,2023-02-16,false,2026-04-29T07:27:21Z,/Volumes/workspace/default/raw_files/retail_store_sales.csv,12575
TXN_7985058,CUST_06,Food,Item_17_FOOD,29.0,5.0,145.0,Cash,In-store,2023-02-23,false,2026-04-29T07:27:21Z,/Volumes/workspace/default/raw_files/retail_store_sales.csv,12575
TXN_5151843,CUST_16,Furniture,Item_19_FUR,32.0,2.0,64.0,Cash,Online,2023-02-09,true,2026-04-29T07:27:21Z,/Volumes/workspace/default/raw_files/retail_store_sales.csv,12575


+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                                                                                                           |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|transaction_id              |string                                                                                                                                                                                              |NULL   |
|customer_id                 |string                    